# Control de bateria de 30 personas

Un grupo de 30 personas sale de paseo y revisa la bateria del celular tres veces en el día: a las 10am, a las 12pm y a las 2pm. Cada persona necesita un mínimo distinto de bateria para llegar a casa sin quedarse incomunicada.

El programa revisa las 90 mediciones (30 personas por 3 chequeos), decide en qué estado está cada una y muestra una tabla con un resumen final.

Este notebook arma el programa por partes. La explicación completa, con la comparación contra el pseudocódigo de PSeInt y la versión en Rust, está en el [README](../README.md) de la carpeta.

## 1. Los datos

Se necesitan tres estructuras:

| Variable | Qué es | Cómo se ve |
|---|---|---|
| `horas` | Vector de 3 textos | `["10am", "12pm", "2pm"]` |
| `minimos` | Vector de 30 enteros | `minimos[i]` es el mínimo de la persona `i` |
| `baterias` | Matriz de 30 filas por 3 columnas | `baterias[i][j]` es la bateria de la persona `i` en el chequeo `j` |

Escribir a mano 90 niveles de bateria sería repetir 90 números en cada lenguaje, así que se calculan con dos fórmulas fijas. No hay azar: las fórmulas dan siempre los mismos números, y por eso las versiones de PSeInt, Python y Rust muestran exactamente lo mismo.

Dos detalles de sintaxis en la celda siguiente:

- `[0] * 30` crea un vector de 30 ceros. `[[0] * 3 for _ in range(30)]` crea 30 filas de 3 ceros cada una, que es la matriz.
- Las posiciones empiezan en **0**: la primera persona es `baterias[0]` y la última es `baterias[29]`. Se recorren con `range(0, 29 + 1)`, donde el `+ 1` recuerda que `range` no incluye el último número.
- `%` es el residuo de la división. `(i * 7) % 31` nunca pasa de 30, y así el mínimo queda entre 15 y 45.

In [21]:
horas = ["10am", "12pm", "2pm"]
minimos = [0] * 30
baterias = [[0] * 3 for _ in range(30)]

for i in range(0, 29 + 1):
    minimos[i] = 15 + ((i * 7) % 31)
    for j in range(0, 2 + 1):
        baterias[i][j] = 10 + ((i * 17 + j * 29) % 90)

print("Personas cargadas   :", len(baterias))
print("Chequeos por persona:", len(baterias[0]))
print("Persona  1 ->", baterias[0], "con minimo", minimos[0])
print("Persona  2 ->", baterias[1], "con minimo", minimos[1])
print("Persona 30 ->", baterias[29], "con minimo", minimos[29])

Personas cargadas   : 30
Chequeos por persona: 3
Persona  1 -> [10, 39, 68] con minimo 15
Persona  2 -> [27, 56, 85] con minimo 22
Persona 30 -> [53, 82, 21] con minimo 32


## 2. La función

Una función es un bloque de código con nombre, que recibe datos y devuelve un resultado. Sirve para no repetir la misma decisión 90 veces.

```
def  nombre_de_la_funcion(parametro1, parametro2):
     |          |              |
     |          |              +-- los datos que entran
     |          +-- el nombre con el que se llama
     +-- la palabra que declara una función
```

`estado_bateria` recibe dos enteros —el nivel de bateria y el mínimo de esa persona— y devuelve un texto con uno de los tres estados:

1. `TRANQUILO` si el nivel es 50 o más.
2. `CUIDADO` si no llega a 50 pero alcanza el mínimo de esa persona.
3. `PELIGRO` si ni siquiera alcanza su mínimo.

`return` termina la función y entrega el valor: en cuanto se ejecuta, las demás líneas ya no se revisan. En Python no se declara el tipo de los parámetros ni el del resultado; en Rust y en PSeInt sí hay que declararlos.

In [22]:
def estado_bateria(nivel, minimo):
    if nivel >= 50:
        return "TRANQUILO"
    elif nivel >= minimo:
        return "CUIDADO"
    else:
        return "PELIGRO"


print("nivel 80, minimo 30 ->", estado_bateria(80, 30))
print("nivel 40, minimo 30 ->", estado_bateria(40, 30))
print("nivel 40, minimo 45 ->", estado_bateria(40, 45))
print("nivel 10, minimo 30 ->", estado_bateria(10, 30))

nivel 80, minimo 30 -> TRANQUILO
nivel 40, minimo 30 -> CUIDADO
nivel 40, minimo 45 -> PELIGRO
nivel 10, minimo 30 -> PELIGRO


## 3. El ciclo anidado y los contadores

Para revisar a todas las personas en todos los chequeos hace falta un ciclo dentro de otro:

- El ciclo de afuera avanza por las **personas** (`i`).
- El ciclo de adentro avanza por los **chequeos** de esa persona (`j`).

Por cada vuelta del ciclo de afuera, el de adentro da tres vueltas completas: 30 por 3 son 90 revisiones.

Un **contador** es una variable que empieza en 0 y crece de a uno dentro del ciclo. `tranquilos = tranquilos + 1` significa "toma el valor que tenía y guárdale uno más". Es la forma de responder cuántos casos hubo de cada estado sin contarlos a mano.

La celda siguiente hace el recorrido solo con las tres primeras personas, para poder ver el detalle de las nueve revisiones.

In [23]:
tranquilos = 0
cuidados = 0
peligros = 0

for i in range(0, 2 + 1):
    for j in range(0, 2 + 1):
        estado = estado_bateria(baterias[i][j], minimos[i])
        print("persona", i + 1, "- chequeo", horas[j], "- bateria", baterias[i][j], "->", estado)
        if estado == "TRANQUILO":
            tranquilos = tranquilos + 1
        elif estado == "CUIDADO":
            cuidados = cuidados + 1
        else:
            peligros = peligros + 1

print("")
print("TRANQUILO:", tranquilos, "  CUIDADO:", cuidados, "  PELIGRO:", peligros)

persona 1 - chequeo 10am - bateria 10 -> PELIGRO
persona 1 - chequeo 12pm - bateria 39 -> CUIDADO
persona 1 - chequeo 2pm - bateria 68 -> TRANQUILO
persona 2 - chequeo 10am - bateria 27 -> CUIDADO
persona 2 - chequeo 12pm - bateria 56 -> TRANQUILO
persona 2 - chequeo 2pm - bateria 85 -> TRANQUILO
persona 3 - chequeo 10am - bateria 44 -> CUIDADO
persona 3 - chequeo 12pm - bateria 73 -> TRANQUILO
persona 3 - chequeo 2pm - bateria 12 -> PELIGRO

TRANQUILO: 4   CUIDADO: 3   PELIGRO: 2


## 4. El programa completo

Esta celda junta todo y produce la salida final. Dos detalles nuevos de sintaxis:

- `print(texto, end="")` imprime **sin** saltar de línea, para ir armando una fila de la tabla por pedazos. El `print("")` del final es el que cierra la fila.
- `f"{valor:>4}"` es un *f-string*: el texto entre comillas donde `{ }` se reemplaza por el valor. El `>4` alinea a la derecha en un ancho de 4 caracteres, y es lo que mantiene las columnas parejas. En Rust se escribe igual, `{:>4}`, dentro de `println!`. En PSeInt no existe: allí hay que alinear a mano con espacios.

Al final, los contadores dan el resumen del día.

In [24]:
def estado_bateria(nivel, minimo):
    if nivel >= 50:
        return "TRANQUILO"
    elif nivel >= minimo:
        return "CUIDADO"
    else:
        return "PELIGRO"


linea_doble = "================================================================="
linea_simple = "-----------------------------------------------------------------"

horas = ["10am", "12pm", "2pm"]
minimos = [0] * 30
baterias = [[0] * 3 for _ in range(30)]
estados = [""] * 3

for i in range(0, 29 + 1):
    minimos[i] = 15 + ((i * 7) % 31)
    for j in range(0, 2 + 1):
        baterias[i][j] = 10 + ((i * 17 + j * 29) % 90)

print(linea_doble)
print("              CONTROL DE BATERIA - GRUPO DE 30 PERSONAS")
print(linea_doble)
print(f"Chequeos del dia: {horas[0]}, {horas[1]}, {horas[2]}")
print("Estados: TRANQUILO (50% o mas) - CUIDADO (alcanza su minimo) - PELIGRO")
print("")
print("No.  Min  10am  12pm   2pm   Estados")
print(linea_simple)

tranquilos = 0
cuidados = 0
peligros = 0
final_en_peligro = 0

for i in range(0, 29 + 1):
    print(f"{i + 1:>3}  {minimos[i]:>2}%", end="")
    for j in range(0, 2 + 1):
        estados[j] = estado_bateria(baterias[i][j], minimos[i])
        print(f"  {baterias[i][j]:>4}", end="")
        if estados[j] == "TRANQUILO":
            tranquilos = tranquilos + 1
        elif estados[j] == "CUIDADO":
            cuidados = cuidados + 1
        else:
            peligros = peligros + 1
    print("   ", end="")
    for j in range(0, 2 + 1):
        print(estados[j], end="")
        if j < 2:
            print(" / ", end="")
    print("")
    if estados[2] == "PELIGRO":
        final_en_peligro = final_en_peligro + 1

print(linea_simple)
print("RESUMEN DEL DIA")
print(f"   Chequeos revisados : {tranquilos + cuidados + peligros}")
print(f"   TRANQUILO          : {tranquilos}")
print(f"   CUIDADO            : {cuidados}")
print(f"   PELIGRO            : {peligros}")
print(f"   Personas que terminan el dia en PELIGRO: {final_en_peligro}")
print(linea_doble)

              CONTROL DE BATERIA - GRUPO DE 30 PERSONAS
Chequeos del dia: 10am, 12pm, 2pm
Estados: TRANQUILO (50% o mas) - CUIDADO (alcanza su minimo) - PELIGRO

No.  Min  10am  12pm   2pm   Estados
-----------------------------------------------------------------
  1  15%    10    39    68   PELIGRO / CUIDADO / TRANQUILO
  2  22%    27    56    85   CUIDADO / TRANQUILO / TRANQUILO
  3  29%    44    73    12   CUIDADO / TRANQUILO / PELIGRO
  4  36%    61    90    29   TRANQUILO / TRANQUILO / PELIGRO
  5  43%    78    17    46   TRANQUILO / PELIGRO / CUIDADO
  6  19%    95    34    63   TRANQUILO / CUIDADO / TRANQUILO
  7  26%    22    51    80   PELIGRO / TRANQUILO / TRANQUILO
  8  33%    39    68    97   CUIDADO / TRANQUILO / TRANQUILO
  9  40%    56    85    24   TRANQUILO / TRANQUILO / PELIGRO
 10  16%    73    12    41   TRANQUILO / PELIGRO / CUIDADO
 11  23%    90    29    58   TRANQUILO / CUIDADO / TRANQUILO
 12  30%    17    46    75   PELIGRO / CUIDADO / TRANQUILO
 13  37%    3

## Retos para practicar

1. Cambiar el 50 de la función por 60 y ver cuántos `TRANQUILO` quedan.
2. Hacer que la tabla muestre solo las personas que terminan el día en `PELIGRO`.
3. Agregar un cuarto chequeo (columna) a las 4pm y ajustar lo que haga falta.
4. Cambiar la función para que devuelva un número (2, 1 o 0) en lugar de un texto, y sumar ese número para obtener un puntaje del día.